# Breakout / trend-following system on a halal universe

This is a different kind of backtest from the other notebooks in this project: instead of
cross-sectionally ranking the halal universe once a month, it trades each stock independently on
its own hourly-bar signal, in the spirit of the classic Donchian-channel "Turtle Trading" system.
Tickers come from `equities-list.csv` (rows with `Asset Type == Stock`; the ETFs in that file are
excluded). The CSV labels these as halal *candidates* only — independently verify current Shariah
compliance before investing.

**Strategy specification**

- **Entry** — buy when a stock's close breaks above its own trailing 55-day high:
  $$ \text{Close}_t > \max(\text{High}_{t-1}, \dots, \text{High}_{t-55}) $$
- **Stop** — a protective stop set at entry, 3&times; the 20-day Average True Range below the
  entry price: $ \text{Stop} = \text{Entry} - 3 \times \text{ATR}_{20} $. This stop is fixed at
  entry, not trailed.
- **Exit** — sell if the stop is hit intraday (trailing stop breach) *or* the close falls below
  the trailing 20-day low, whichever comes first:
  $$ \text{Close}_t < \min(\text{Low}_{t-1}, \dots, \text{Low}_{t-20}) $$
- **Position management** — long only, one position per name at a time (no pyramiding/adding to
  winners), up to 15 concurrent positions, each sized as an equal share of current portfolio value
  at the moment it's entered. If more than 15 stocks would qualify to enter on the same day, the
  strongest breakouts (furthest above their 55-day high, as a fraction) get the open slots first.

This is a genuinely different backtest engine from the cross-sectional notebooks: trades happen on
whatever hourly bar a signal fires, not on a monthly calendar, so this notebook runs an hourly
event-driven simulation rather than a monthly rebalance loop. Every rule above is evaluated on each
hourly bar; the channel and ATR windows keep their trading-day lengths (55 / 20 / 20 sessions),
expressed in bars at 7 per session, and the hourly OHLC bars come from the shared
`data/hourly/{TICKER}.csv` cache.


## Before running

1. Install dependencies: `pip install ib-insync pandas numpy matplotlib pandas-datareader`
2. Run IBKR Trader Workstation or IB Gateway, enable API connections, and use a market-data-enabled
   account.
3. Set the host, port, and client ID below. Typical paper ports are 7497 for TWS and 4002 for
   Gateway; live ports are commonly 7496 and 4001.
4. Run the acquisition cell once. Each ticker's hourly bars are cached to
   `data/hourly/{TICKER}.csv`, and a summary of what downloaded is written to
   `data/hourly_download_log.csv`. All seven strategy notebooks share this cache. Re-running skips
   tickers that are already cached and logged `ok`, so it is safe to re-run if it is interrupted
   partway through.
5. IBKR only serves a limited span per hourly request, so each ticker is fetched backward in
   1-year chunks (about 18 requests per ticker back to 2009, roughly 2,400 for the full
   universe). With the default sleeps between requests, expect the first full download to take
   on the order of two hours; every notebook after that reads straight from the cache.


In [ ]:
from pathlib import Path
import csv
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from ib_insync import IB, Stock, util
except ImportError as exc:
    raise ImportError('Install ib-insync first: pip install ib-insync') from exc

EQUITIES_LIST_PATH = Path('equities-list.csv')
DATA_DIR = Path('data') / 'hourly'
LOG_PATH = Path('data') / 'hourly_download_log.csv'

with open(EQUITIES_LIST_PATH, newline='') as f:
    equities_rows = list(csv.DictReader(f))
UNIVERSE = [row['Ticker'] for row in equities_rows if row['Asset Type'] == 'Stock']
print(f'{len(UNIVERSE)} candidate halal stocks in the universe')

EXCHANGE = 'SMART'
CURRENCY = 'USD'
START_DATE = pd.Timestamp('2009-01-01', tz='UTC')
END_DATE = pd.Timestamp.now(tz='UTC')
BAR_SIZE = '1 hour'
DURATION = '1 Y'  # hourly history is fetched in 1-year chunks, walking backward from today
WHAT_TO_SHOW = 'TRADES'
USE_RTH = True

IB_HOST = '172.27.224.1'  # '127.0.0.1'
IB_PORT = 7497
IB_CLIENT_ID = 35
PACING_SLEEP_SECONDS = 3.0   # between tickers
CHUNK_SLEEP_SECONDS = 1.5    # between 1-year chunk requests
# Hourly bar geometry: US regular hours = 09:30 half-hour bar + six 1-hour bars = 7 bars/session.
BARS_PER_SESSION = 7
TRADING_DAYS_PER_YEAR = 252
BARS_PER_YEAR = BARS_PER_SESSION * TRADING_DAYS_PER_YEAR   # annualization factor for hourly returns
# Look-backs are written in trading days (each strategy's real-time horizon) and converted to bars
# with this factor. Set it to 1 to read every "day" window as a raw bar count instead (e.g. a
# 200-bar SMA), which turns the strategy into a much faster, shorter-horizon variant.
WINDOW_BARS_PER_DAY = BARS_PER_SESSION

# Strategy parameters
ENTRY_LOOKBACK_DAYS = 55   # breakout channel, in trading days
EXIT_LOOKBACK_DAYS = 20    # trend-exit channel, in trading days
ATR_PERIOD = 20            # ATR window, in trading days (measured on daily ranges rebuilt from hourly bars)
ENTRY_LOOKBACK_BARS = ENTRY_LOOKBACK_DAYS * WINDOW_BARS_PER_DAY
EXIT_LOOKBACK_BARS = EXIT_LOOKBACK_DAYS * WINDOW_BARS_PER_DAY
ATR_STOP_MULT = 3.0        # stop distance, in ATRs below entry
MAX_POSITIONS = 15         # concurrent position cap

INVESTMENT_2026_DOLLARS = 150.0
TARGET_CPI = None  # leave None to use the latest available CPI reading from FRED

## Download or load the hourly bars

Hourly bars (unlike daily bars) can't be requested many years back in a single IBKR call, so each
ticker is walked backward in 1-year chunks from today until `START_DATE` or until IBKR has no
earlier data (a recent IPO simply stops early — check `first_date` in the download log). The
function still checks the on-disk cache first and only opens an IBKR connection if at least one
ticker's cache is missing. Bars are regular-trading-hours only: a 09:30 half-hour bar plus six
1-hour bars, i.e. 7 bars per full session.


In [ ]:
def normalize_bars(bars):
    frame = util.df(bars)
    if frame is None or frame.empty:
        return pd.DataFrame()
    frame['date'] = pd.to_datetime(frame['date'], utc=True)
    frame = frame.set_index('date').sort_index()
    keep = ['open', 'high', 'low', 'close', 'volume', 'average', 'barCount']
    return frame[[column for column in keep if column in frame.columns]]

def load_cached_bars(ticker):
    frame = pd.read_csv(DATA_DIR / f'{ticker}.csv', parse_dates=['date'], index_col='date')
    frame.index = pd.DatetimeIndex(frame.index).tz_convert('UTC')
    return frame.sort_index()

def fetch_ticker_hourly(ib, contract):
    """Walk backward in 1-year chunks until START_DATE or until IBKR has no earlier bars."""
    chunks, cursor, empty_in_a_row = [], END_DATE, 0
    while cursor > START_DATE:
        bars = ib.reqHistoricalData(
            contract, endDateTime=cursor.strftime('%Y%m%d %H:%M:%S UTC'), durationStr=DURATION,
            barSizeSetting=BAR_SIZE, whatToShow=WHAT_TO_SHOW,
            useRTH=USE_RTH, formatDate=2, keepUpToDate=False,
        )
        chunk = normalize_bars(bars)
        if chunk.empty:
            empty_in_a_row += 1
            if chunks or empty_in_a_row >= 2:  # ran out of history, or nothing recent at all
                break
            cursor -= pd.Timedelta(days=365)
        else:
            chunks.append(chunk)
            empty_in_a_row = 0
            cursor = chunk.index.min() - pd.Timedelta(hours=1)
        time.sleep(CHUNK_SLEEP_SECONDS)
    if not chunks:
        return pd.DataFrame()
    result = pd.concat(chunks).sort_index()
    result = result.loc[~result.index.duplicated(keep='last')]
    return result.loc[result.index >= START_DATE]

def download_universe():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    util.patchAsyncio()
    ib = IB()
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID, readonly=True, timeout=20)

    log_rows = list(csv.DictReader(open(LOG_PATH))) if LOG_PATH.exists() else []
    done = {row['ticker'] for row in log_rows if row['status'] == 'ok'}

    for i, ticker in enumerate(UNIVERSE, 1):
        cache_path = DATA_DIR / f'{ticker}.csv'
        if ticker in done and cache_path.exists():
            continue
        status, n_bars, first_date, last_date, error = 'ok', 0, '', '', ''
        try:
            contract = Stock(ticker, EXCHANGE, CURRENCY)
            qualified = ib.qualifyContracts(contract)
            if not qualified or contract.conId == 0:
                raise RuntimeError('contract could not be qualified')
            frame = fetch_ticker_hourly(ib, contract)
            if frame.empty:
                status = 'no_data'
            else:
                frame.to_csv(cache_path, index_label='date')
                n_bars = len(frame)
                first_date, last_date = str(frame.index.min().date()), str(frame.index.max().date())
        except Exception as exc:
            status, error = 'error', str(exc)
        print(f'[{i}/{len(UNIVERSE)}] {ticker}: {status} n_bars={n_bars} {first_date}..{last_date} {error}')
        log_rows = [r for r in log_rows if r['ticker'] != ticker]
        log_rows.append({'ticker': ticker, 'status': status, 'n_bars': n_bars,
                          'first_date': first_date, 'last_date': last_date, 'error': error})
        with open(LOG_PATH, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['ticker', 'status', 'n_bars', 'first_date', 'last_date', 'error'])
            writer.writeheader()
            writer.writerows(log_rows)
        time.sleep(PACING_SLEEP_SECONDS)

    ib.disconnect()

missing = [t for t in UNIVERSE if not (DATA_DIR / f'{t}.csv').exists()]
if missing:
    print(f'{len(missing)} tickers not yet cached, downloading from IBKR...')
    download_universe()
else:
    print('All tickers already cached.')


## Build the OHLC panels and the signals

Three wide hourly panels (high, low, close), then the Donchian channels and ATR computed
vectorized across the whole universe at once. Channel windows are the same number of trading days as
before, expressed in bars (55 sessions = 385 bars, 20 sessions = 140 bars):

- `donchian_high` — the trailing 55-day (385-bar) high, shifted by one bar so an entry decision only
  ever looks at *prior* bars.
- `donchian_low` — the trailing 20-day (140-bar) low, same shift, used for the trend-exit.
- `atr` — 20-day average true range, taken from daily sessions rebuilt out of the hourly bars (the
  standard three-way true range) and lagged one session, so the 3&times;ATR stop sits at the same
  distance as in the daily system. What changes is that the stop is now watched on every hourly bar.
- `entry_signal` — this bar's close above the (shifted) 55-day high.


In [ ]:
hourly_bars = {}
for ticker in UNIVERSE:
    cache_path = DATA_DIR / f'{ticker}.csv'
    if not cache_path.exists():
        continue
    bars = load_cached_bars(ticker)
    if bars.empty:
        continue
    hourly_bars[ticker] = bars

high_panel = pd.DataFrame({t: b['high'] for t, b in hourly_bars.items()}).sort_index()
low_panel = pd.DataFrame({t: b['low'] for t, b in hourly_bars.items()}).sort_index()
close_panel = pd.DataFrame({t: b['close'] for t, b in hourly_bars.items()}).sort_index()

high_panel = high_panel.loc[high_panel.index >= START_DATE]
low_panel = low_panel.reindex(high_panel.index)
close_panel = close_panel.reindex(high_panel.index)

# Fill isolated missing bars (at most one session) so bar-count rolling windows stay contiguous.
close_panel = close_panel.ffill(limit=BARS_PER_SESSION)
high_panel = high_panel.fillna(close_panel)
low_panel = low_panel.fillna(close_panel)

print(f'{close_panel.shape[1]} tickers, {close_panel.shape[0]:,} hourly bars '
      f'from {close_panel.index.min().date()} to {close_panel.index.max().date()}')

# ATR: true range measured on daily sessions rebuilt from the hourly bars, so the 3x ATR stop keeps
# the same distance as the daily system; the stop is then checked against every hourly bar's low.
# A session's ATR is only known once it has closed, so each bar uses the previous session's ATR.
session = close_panel.index.tz_convert('America/New_York').tz_localize(None).normalize()
daily_high = high_panel.groupby(session).max()
daily_low = low_panel.groupby(session).min()
daily_close = close_panel.groupby(session).last()
prev_daily_close = daily_close.shift(1)
true_range = np.maximum(np.maximum(daily_high - daily_low, (daily_high - prev_daily_close).abs()),
                         (daily_low - prev_daily_close).abs())
atr_daily = true_range.rolling(ATR_PERIOD, min_periods=ATR_PERIOD).mean().shift(1)
atr = atr_daily.reindex(session).set_axis(close_panel.index, axis=0)

donchian_high = high_panel.rolling(ENTRY_LOOKBACK_BARS, min_periods=ENTRY_LOOKBACK_BARS).max().shift(1)
donchian_low = low_panel.rolling(EXIT_LOOKBACK_BARS, min_periods=EXIT_LOOKBACK_BARS).min().shift(1)
entry_signal = close_panel > donchian_high


## Hourly event-driven backtest

Unlike the monthly notebooks, this walks forward one hourly bar at a time: process exits first (a
stop-loss breach takes priority over a channel exit on the same bar), mark the portfolio to market,
then fill any open slots with the bar's strongest new breakouts. Converting the signal frames to
plain numpy arrays first keeps ~31,000 bars x ~130 tickers manageable.


In [ ]:
dates = close_panel.index
tickers = close_panel.columns.tolist()
ticker_idx = {t: j for j, t in enumerate(tickers)}
n_dates = len(dates)

close_v = close_panel.values
low_v = low_panel.values
atr_v = atr.values
donch_low_v = donchian_low.values
donch_high_v = donchian_high.values
entry_v = entry_signal.values

positions = {}
cash = 1.0
nav_history = np.empty(n_dates)
trade_log = []

for i in range(n_dates):
    # 1. Exits: a stop-loss breach takes priority over a channel exit the same day.
    to_remove = []
    for tkr, pos in positions.items():
        j = ticker_idx[tkr]
        c = close_v[i, j]
        if np.isnan(c):
            continue
        lo = low_v[i, j]
        exit_price, exit_reason = None, None
        if not np.isnan(lo) and lo <= pos['stop']:
            exit_price, exit_reason = pos['stop'], 'stop'
        else:
            dlow = donch_low_v[i, j]
            if not np.isnan(dlow) and c < dlow:
                exit_price, exit_reason = c, 'channel'
        if exit_price is not None:
            cash += pos['shares'] * exit_price
            trade_log.append({
                'ticker': tkr, 'entry_date': pos['entry_date'], 'entry_price': pos['entry_price'],
                'exit_date': dates[i], 'exit_price': exit_price, 'exit_reason': exit_reason,
                'return_pct': (exit_price / pos['entry_price'] - 1.0) * 100,
                'holding_days': (dates[i] - pos['entry_date']).total_seconds() / 86400,
            })
            to_remove.append(tkr)
    for tkr in to_remove:
        del positions[tkr]

    open_value = sum(positions[t]['shares'] * close_v[i, ticker_idx[t]] for t in positions
                      if not np.isnan(close_v[i, ticker_idx[t]]))
    nav_at_entry_phase = cash + open_value

    # 2. Entries: rank same-day candidates by breakout strength, fill open slots.
    free_slots = MAX_POSITIONS - len(positions)
    if free_slots > 0:
        candidates = []
        for j, tkr in enumerate(tickers):
            if tkr in positions:
                continue
            if (entry_v[i, j] and not np.isnan(atr_v[i, j]) and not np.isnan(close_v[i, j])
                    and close_v[i, j] > 0 and atr_v[i, j] > 0):
                strength = close_v[i, j] / donch_high_v[i, j] - 1.0
                candidates.append((strength, tkr, j))
        candidates.sort(reverse=True)
        for strength, tkr, j in candidates[:free_slots]:
            if cash <= 0:
                break  # no cash left to deploy this day, regardless of open slots
            alloc = min(nav_at_entry_phase / MAX_POSITIONS, cash)
            entry_price = close_v[i, j]
            shares = alloc / entry_price
            stop = entry_price - ATR_STOP_MULT * atr_v[i, j]
            positions[tkr] = {'shares': shares, 'stop': stop, 'entry_price': entry_price, 'entry_date': dates[i]}
            cash -= alloc

    open_value = sum(positions[t]['shares'] * close_v[i, ticker_idx[t]] for t in positions
                      if not np.isnan(close_v[i, ticker_idx[t]]))
    nav_history[i] = cash + open_value

nav = pd.Series(nav_history, index=dates)
trades = pd.DataFrame(trade_log)
print(f'{len(trades):,} completed trades, {len(positions)} open positions as of {dates[-1].date()}')

### Current open positions

Whatever the system is holding as of the last cached trading day — the same shape as the "latest
holdings" table in the other notebooks, but here it's a live snapshot of an always-on system
rather than a monthly rebalance.

In [ ]:
last_i = n_dates - 1
open_rows = []
for tkr, pos in positions.items():
    j = ticker_idx[tkr]
    current_price = close_v[last_i, j]
    open_rows.append({
        'ticker': tkr,
        'entry_date': pos['entry_date'].date(),
        'entry_price': round(pos['entry_price'], 2),
        'current_price': round(current_price, 2) if not np.isnan(current_price) else None,
        'unrealized_pct': round((current_price / pos['entry_price'] - 1) * 100, 1) if not np.isnan(current_price) else None,
        'stop_price': round(pos['stop'], 2),
        'pct_to_stop': round((current_price / pos['stop'] - 1) * 100, 1) if not np.isnan(current_price) else None,
    })
open_positions_table = pd.DataFrame(open_rows)
if not open_positions_table.empty:
    open_positions_table = open_positions_table.sort_values('unrealized_pct', ascending=False)
display(open_positions_table)

## Trade statistics

This is "more of a trading strategy," so its natural scoreboard is trade-level, not just monthly
returns: how many trades, how often they win, and by how much when they do versus when they
don't. Classic trend-following systems tend to have a win rate under 50% with a much larger
average winner than average loser — that's the payoff shape to look for here, not a high hit
rate.

In [ ]:
trade_stats = pd.Series({
    'total_trades': len(trades),
    'win_rate': float((trades['return_pct'] > 0).mean()),
    'avg_win_pct': float(trades.loc[trades['return_pct'] > 0, 'return_pct'].mean()),
    'avg_loss_pct': float(trades.loc[trades['return_pct'] <= 0, 'return_pct'].mean()),
    'avg_holding_days': float(trades['holding_days'].mean()),
    'best_trade_pct': float(trades['return_pct'].max()),
    'worst_trade_pct': float(trades['return_pct'].min()),
    'profit_factor': float(
        trades.loc[trades['return_pct'] > 0, 'return_pct'].sum()
        / -trades.loc[trades['return_pct'] <= 0, 'return_pct'].sum()
    ),
})
display(trade_stats.to_frame('value'))
display(trades['exit_reason'].value_counts().to_frame('trade_count'))

## General statistics

Same shape of summary as the other notebooks, computed from the hourly NAV series (annualized with &radic;(252 x 7) bars per year).

In [ ]:
def max_drawdown(wealth):
    drawdown = wealth / wealth.cummax() - 1.0
    return float(drawdown.min())

nav_returns = nav.pct_change().dropna()
start_date, end_date = nav.index[0], nav.index[-1]
years = (end_date - start_date).days / 365.2425
total_return = float(nav.iloc[-1] - 1.0)
cagr = float(nav.iloc[-1] ** (1 / years) - 1.0)
annualized_volatility = float(nav_returns.std() * np.sqrt(BARS_PER_YEAR))
sharpe_rf0 = float((nav_returns.mean() * BARS_PER_YEAR) / annualized_volatility) if annualized_volatility else float('nan')

stats = pd.Series({
    'start': start_date,
    'end': end_date,
    'hourly_bars': len(nav),
    'total_return': total_return,
    'CAGR': cagr,
    'annualized_volatility': annualized_volatility,
    'sharpe_ratio_rf0': sharpe_rf0,
    'max_drawdown': max_drawdown(nav),
    'final_wealth_multiple': float(nav.iloc[-1]),
})
display(stats.to_frame('value'))

## Buy-and-hold-style dollar simulation

Same purchasing-power framing as the other notebooks: size a 2009-era investment so it has the
same purchasing power as $150 today (via FRED CPI), run it through this strategy's compounded
return, and compare the ending value against today's $150 in real terms. This is a price-only,
no-cost simulation, and this notebook in particular assumes stop and channel exits fill at exactly
their trigger price with no slippage — see caveats.

In [ ]:
try:
    from pandas_datareader import data as web
    cpi = web.DataReader('CPIAUCSL', 'fred', '2008-12-01', pd.Timestamp.now())['CPIAUCSL'].dropna()
    cpi_2009 = float(cpi.loc[cpi.index >= '2009-01-01'].iloc[0])
    target_cpi = float(TARGET_CPI) if TARGET_CPI is not None else float(cpi.iloc[-1])
    cpi_source = 'FRED CPIAUCSL'
except Exception as exc:
    if TARGET_CPI is None:
        raise RuntimeError('FRED CPI retrieval failed. Set TARGET_CPI to a CPIAUCSL value for the target dollar period.') from exc
    cpi_2009 = 211.143
    target_cpi = float(TARGET_CPI)
    cpi_source = 'manual target CPI; Jan 2009 CPI fallback'

initial_2009_dollars = INVESTMENT_2026_DOLLARS * cpi_2009 / target_cpi
ending_value = initial_2009_dollars * nav.iloc[-1]
nominal_gain = ending_value - initial_2009_dollars
real_gain_vs_target_dollars = ending_value - INVESTMENT_2026_DOLLARS
real_multiple_vs_target_dollars = ending_value / INVESTMENT_2026_DOLLARS

simulation = pd.Series({
    'target_dollars': INVESTMENT_2026_DOLLARS,
    'cpi_2009': cpi_2009,
    'target_cpi': target_cpi,
    'initial_investment_in_2009_dollars': initial_2009_dollars,
    'ending_nominal_value': ending_value,
    'nominal_gain': nominal_gain,
    'total_return': total_return,
    'CAGR': cagr,
    'ending_value_in_target_dollars': ending_value,
    'real_gain_vs_target_dollars': real_gain_vs_target_dollars,
    'real_multiple_vs_target_dollars': real_multiple_vs_target_dollars,
})
print(f'CPI source: {cpi_source}')
display(simulation.to_frame('value'))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
nav.plot(ax=ax, color='#176b87', title='Growth of $1 — 55-day breakout / 20-day exit / 3xATR stop, halal universe')
ax.set_ylabel('Portfolio value')
ax.axhline(1, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(13, 3))
open_count = pd.Series(0, index=dates)
for t in trades.itertuples():
    open_count.loc[t.entry_date:t.exit_date] += 1
open_count.plot(ax=ax, color='#d97706', title=f'Concurrent open positions (cap = {MAX_POSITIONS})')
ax.set_ylabel('Positions held')
plt.tight_layout()
plt.show()

## Monthly performance

The hourly NAV series resampled to month-end, so this strategy can be read on the same monthly
heatmap as the other notebooks even though it actually trades on hourly signals.

In [ ]:
monthly_nav = nav.resample('ME').last()
today_ts = pd.Timestamp.now(tz='UTC')
if len(monthly_nav) and (monthly_nav.index[-1].year, monthly_nav.index[-1].month) == (today_ts.year, today_ts.month):
    monthly_nav = monthly_nav.iloc[:-1]
monthly_returns = monthly_nav.pct_change().dropna()

table = (monthly_returns * 100).rename('ret').to_frame()
table['year'] = table.index.year
table['month'] = table.index.strftime('%b')
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_table = table.pivot(index='year', columns='month', values='ret').reindex(columns=month_order)

annual_return = monthly_returns.groupby(monthly_returns.index.year).apply(lambda x: (1 + x).prod() - 1) * 100
monthly_table['Year'] = annual_return

monthly_summary = pd.Series({
    'months': len(monthly_returns),
    'positive_months': int((monthly_returns > 0).sum()),
    'pct_positive_months': float((monthly_returns > 0).mean()),
    'avg_monthly_return': float(monthly_returns.mean()),
    'median_monthly_return': float(monthly_returns.median()),
    'best_month': monthly_returns.idxmax().strftime('%Y-%m'),
    'best_month_return': float(monthly_returns.max()),
    'worst_month': monthly_returns.idxmin().strftime('%Y-%m'),
    'worst_month_return': float(monthly_returns.min()),
})

styled = (
    monthly_table.round(1)
    .style.format(precision=1, na_rep='')
    .background_gradient(cmap='RdYlGn', axis=None, vmin=-15, vmax=15, subset=month_order)
    .set_caption('Monthly return (%) by year')
)
display(styled)
display(monthly_summary.to_frame('value'))

In [ ]:
colors = np.where(monthly_returns >= 0, '#176b87', '#c0392b')
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(monthly_returns.index, monthly_returns.values * 100, width=20, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Breakout system monthly return (%)')
ax.set_ylabel('Return (%)')
plt.tight_layout()
plt.show()

## Caveats and next improvements

- **No slippage or gaps**: stops and channel exits are assumed to fill at exactly their trigger
  price. In practice a stock gapping down through its stop overnight fills *worse* than the stop
  price — this is the single biggest way this backtest is optimistic, more so than the monthly
  notebooks, since a fast-moving breakout/breakdown system is exactly the kind of strategy gap
  risk hurts most.
- **No transaction costs**: 1,300+ trades over the backtest window means commissions and spread
  matter a lot here — more trades than any of the cross-sectional notebooks.
- **Same-bar signal and fill**: entry uses today's close to both trigger *and* fill the trade,
  and exits similarly assume the trigger price is achievable. A live system only knows the close
  once the market has closed.
- **No pyramiding**: real trend-following systems (including the original Turtle rules) often add
  to a winning position as it extends; this notebook opens exactly one unit per name and lets it
  ride to its exit.
- **Equal-weight-of-NAV sizing, not volatility-based**: the classic Turtle system sizes each entry
  so that hitting the stop loses a fixed fraction of capital (a smaller position in a more volatile
  name). This notebook instead gives every one of the 15 slots an equal dollar amount regardless of
  ATR, which is simpler but means a volatile breakout risks more capital than a quiet one. Sizing is
  cash-constrained — a new entry never spends more than what's actually sitting in cash, even if the
  1/15-of-NAV target momentarily exceeds it (e.g. right after existing positions have run up).
- **Fixed, non-trailing stop**: the stop is set once at entry and never moves up as a position
  gains. A trailing-stop variant (recompute `close - 3*ATR` and only ratchet it upward) is a
  common, easy extension.
- **20-day-low exit only**: the alternative mentioned in the original spec — exiting on a break
  below the 50-day moving average — isn't implemented here, but would be a one-line swap for the
  channel-exit condition.
- **Look-ahead / survivorship bias in the universe**: `equities-list.csv` is today's list of halal
  candidates, projected backward, same caveat as every other notebook in this project.
- **Hourly bars, still price-only and still costless**: hourly bars make signals and marks finer,
  not fills more realistic. Every entry and exit here still fills at the bar's own close (or the
  trigger price), with no spread, slippage or commissions — and an hourly system that reacts to
  more bars trades more, so the costs left out matter more than in the daily version. Look-backs are
  written in trading days and converted at 7 bars per session; half-day sessions have fewer bars,
  so those windows stretch slightly in calendar time.
- A serious version of this strategy would need realistic slippage assumptions (especially around
  gap risk), volatility-based position sizing, and probably a comparison of trailing vs. fixed
  stops before trusting the backtest's drawdown numbers.